# Medical Transcription Specialty Classifier

**Improved Architecture** with LSTM + Attention for better accuracy

In [ ]:
import kagglehub
import os

# Download dataset
path = kagglehub.dataset_download("tboyle10/medicaltranscriptions")
print("Dataset path:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
data = pd.read_csv(os.path.join(path, 'mtsamples.csv'))
print(f"Shape: {data.shape}")
data.head()

In [ ]:
# Check for missing values
print(data.isnull().sum())
print(f"\nUnique specialties: {data['medical_specialty'].nunique()}")
print(f"\nSpecialty distribution:")
print(data['medical_specialty'].value_counts())

## Data Preprocessing

Cleaning text and handling class imbalance

In [ ]:
import re
import string
from nltk.corpus import stopwords
import nltk

# Download NLTK data
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

def clean_text(text):
    '''Clean and preprocess medical text'''
    if not isinstance(text, str):
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove email
    text = re.sub(r'[\w\.-]+@[\w\.-]+', '', text)
    
    # Remove numbers and special chars but keep spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Clean transcriptions
data['transcription_clean'] = data['transcription'].apply(clean_text)

# Remove empty after cleaning
data = data[data['transcription_clean'].str.len() > 50].copy()

print(f"Data after cleaning: {data.shape}")
print(f"\nExample cleaned text:")
print(data['transcription_clean'].iloc[0][:300])

In [ ]:
# Remove rare classes (< 10 samples)
specialty_counts = data['medical_specialty'].value_counts()
valid_specialties = specialty_counts[specialty_counts >= 10].index
data = data[data['medical_specialty'].isin(valid_specialties)].copy()

print(f"Classes after filtering: {data['medical_specialty'].nunique()}")
print(f"Data shape: {data.shape}")

# Visualize class distribution
fig, ax = plt.subplots(figsize=(14, 5))
specialty_counts = data['medical_specialty'].value_counts()
sns.barplot(x=specialty_counts.values, y=specialty_counts.index, ax=ax)
ax.set_xlabel('Number of Reports')
ax.set_title('Medical Specialty Distribution (after filtering)')
plt.tight_layout()
plt.show()

## Tokenization & Encoding

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

VOCAB_SIZE = 15000
MAX_LEN = 400
EMBED_DIM = 100

# Tokenize
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(data['transcription_clean'])

X_seq = tokenizer.texts_to_sequences(data['transcription_clean'])
X_padded = pad_sequences(
    X_seq,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print(f"Padded sequences shape: {X_padded.shape}")
print(f"Vocab size: {len(tokenizer.word_index) + 1}")

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y = data['medical_specialty']
y_encoded = label_encoder.fit_transform(y)
num_classes = len(label_encoder.classes_)

print(f"Number of classes: {num_classes}")
print(f"Classes: {label_encoder.classes_}")

# Compute class weights to handle imbalance
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_encoded),
    y=y_encoded
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f"\nClass weights (sample): {dict(list(class_weight_dict.items())[:3])}")

## Train/Validation/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split 70% train, 15% val, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X_padded, y_encoded,
    test_size=0.15,
    random_state=42,
    stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.18,  # 15% of total
    random_state=42,
    stratify=y_temp
)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

## Advanced Model: Bidirectional LSTM + Multi-Head Attention

This architecture captures long-range dependencies better than simple pooling.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks

def create_model(vocab_size, max_len, num_classes, embed_dim=100):
    '''Build LSTM + Attention model'''
    
    inputs = layers.Input(shape=(max_len,), name='input')
    
    # Embedding layer
    x = layers.Embedding(
        input_dim=vocab_size,
        output_dim=embed_dim,
        name='embedding'
    )(inputs)
    
    x = layers.Dropout(0.3)(x)
    
    # Bidirectional LSTM
    x = layers.Bidirectional(
        layers.LSTM(128, return_sequences=True, dropout=0.3),
        name='bilstm'
    )(x)
    
    # Multi-Head Attention
    attention = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=64,
        dropout=0.2,
        name='attention'
    )
    
    attn_out, attn_scores = attention(
        x, x, return_attention_scores=True
    )
    
    # Add residual connection
    x = layers.Add()([x, attn_out])
    x = layers.LayerNormalization()(x)
    
    # Global pooling
    x = layers.GlobalAveragePooling1D()(x)
    
    # Dense layers
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    
    # Output
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    
    # Also create attention score model
    score_model = models.Model(inputs=inputs, outputs=attn_scores)
    
    return model, score_model

model, score_model = create_model(
    VOCAB_SIZE, MAX_LEN, num_classes, EMBED_DIM
)

model.summary()

## Training with Advanced Callbacks

Using EarlyStopping, ReduceLROnPlateau, and class weights

In [ ]:
# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# Evaluate on test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Per-class accuracy
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
from sklearn.metrics import classification_report, confusion_matrix

print("\n" + "="*60)
print("Classification Report:")
print("="*60)
print(classification_report(
    y_test, y_pred,
    target_names=label_encoder.classes_,
    digits=3
))

In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Model Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Model Interpretability

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
    cmap='Blues'
)
plt.title('Confusion Matrix')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Token importance for a sample
def get_top_tokens(text, tokenizer, label_encoder, model, score_model, top_n=15):
    '''Extract important tokens using attention'''
    
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    
    # Get prediction
    pred = model.predict(padded, verbose=0)[0]
    pred_class = np.argmax(pred)
    specialty = label_encoder.inverse_transform([pred_class])[0]
    confidence = pred[pred_class] * 100
    
    # Get attention scores
    attn = score_model.predict(padded, verbose=0)
    importance = attn.mean(axis=(1, 2))[0]
    
    # Map tokens
    reverse_idx = {v: k for k, v in tokenizer.word_index.items()}
    tokens_list = []
    
    for token_id, score in zip(padded[0], importance):
        if token_id != 0:
            tokens_list.append((
                reverse_idx.get(token_id, ''),
                float(score)
            ))
    
    tokens_list.sort(key=lambda x: x[1], reverse=True)
    
    return specialty, confidence, tokens_list[:top_n], padded

# Example
sample_text = data['transcription_clean'].iloc[10]
specialty, conf, tokens, _ = get_top_tokens(
    sample_text, tokenizer, label_encoder, model, score_model
)

print(f"Text: {sample_text[:200]}...")
print(f"\nPredicted Specialty: {specialty}")
print(f"Confidence: {conf:.2f}%")
print(f"\nTop Important Tokens:")
for word, score in tokens:
    print(f"  {word:20s} -> {score:.6f}")

## Save Model & Artifacts

In [ ]:
import pickle
import os

# Create output directory if it doesn't exist
os.makedirs('model_artifacts', exist_ok=True)

# Save model
model.save('model_artifacts/medical_classifier.h5')
print('✓ Model saved')

# Save tokenizer
with open('model_artifacts/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
print('✓ Tokenizer saved')

# Save label encoder
with open('model_artifacts/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print('✓ Label encoder saved')

# Save vocab size and max len
config = {'VOCAB_SIZE': VOCAB_SIZE, 'MAX_LEN': MAX_LEN, 'EMBED_DIM': EMBED_DIM}
with open('model_artifacts/config.pkl', 'wb') as f:
    pickle.dump(config, f)
print('✓ Config saved')

print(f"\nAll artifacts saved to 'model_artifacts/' directory")